# Minimales `bpy` im Notebook

Dieses Notebook installiert das geprüfte, inoffizielle Minimal-Wheel für Python 3.11 oder 3.12. Es ist ausschließlich für Mesh-/Material-/Hierarchie-/Text-Erzeugung und das Speichern von `.blend`-Dateien vorgesehen. Rendering, Simulation, Audio/Video und fremde Dateiformate gehören nicht zum zugesagten Umfang.

## 1. Projektmodule aus dem Repository installieren

Die Build-Metadaten liegen im Wurzelverzeichnis des Repositories. Deshalb wird hier **kein** `#subdirectory=imports` verwendet.

In [ ]:
%pip install --upgrade --no-cache-dir "hypercube-graph-surface @ git+https://github.com/Hadron-JLeo/BachelorArbeit.git@main"

In [ ]:
from imports import GeometryConfig, Polygon3D, generate_hypercube_surface

assert GeometryConfig is not None
assert Polygon3D is not None
assert callable(generate_hypercube_surface)
print("Projektmodule erfolgreich importiert.")

## 2. Passendes `bpy`-Wheel auswählen und kryptografisch prüfen

In [ ]:
import hashlib
import platform
import sys
import sysconfig
from pathlib import Path
from urllib.request import urlretrieve

assert sys.platform.startswith("linux"), f"Nur Linux wird unterstützt: {sys.platform}"
assert platform.machine() in {"x86_64", "AMD64"}, platform.machine()
libc_name, libc_version = platform.libc_ver()
assert libc_name == "glibc", (libc_name, libc_version)
assert tuple(map(int, libc_version.split(".")[:2])) >= (2, 28), libc_version
python_key = f"{sys.version_info.major}.{sys.version_info.minor}"
soabi = sysconfig.get_config_var("SOABI")
assert soabi and soabi.startswith(f"cpython-{sys.version_info.major}{sys.version_info.minor}"), soabi
wheel_names = {
    "3.11": "bpy-4.5.3+mesh1-cp311-cp311-manylinux_2_28_x86_64.whl",
    "3.12": "bpy-4.5.3+mesh1-cp312-cp312-manylinux_2_28_x86_64.whl",
}
wheel_hashes = {
    "3.11": "651c74dce5c00b1a822c4ff0d78d6d0cf3904079a78308e171acf28e53629409",
    "3.12": "1f26833ce95fd2c60f42d9f44b34a97326a3a6246d0b3d177d59fb83e99e2ff5",
}
if python_key not in wheel_names:
    raise RuntimeError(f"Kein geprüftes Wheel für Python {python_key}; verfügbar: {sorted(wheel_names)}")

wheel_name = wheel_names[python_key]
wheel_url = (
    "https://github.com/Hadron-JLeo/BachelorArbeit/releases/download/"
    f"bpy-mesh-4.5.3.1/{wheel_name}"
)
wheel_path = Path("/tmp") / wheel_name
urlretrieve(wheel_url, wheel_path)
actual_hash = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
assert actual_hash == wheel_hashes[python_key], (actual_hash, wheel_hashes[python_key])
print(f"Geprüft: {wheel_path.name} ({wheel_path.stat().st_size / 1_000_000:.1f} MB)")

In [ ]:
%pip install --no-cache-dir --no-deps --force-reinstall {wheel_path}

Falls in dieser Sitzung vorher bereits ein anderes `bpy` importiert wurde, den Kernel jetzt einmal neu starten. Bei einer frischen Sitzung ist das nicht nötig.

## 3. Kleinen `.blend`-Export ausführen

In [ ]:
from importlib.metadata import version
import bpy

assert version("bpy") == "4.5.3+mesh1"
assert bpy.app.background
bpy.ops.wm.read_factory_settings(use_empty=True)

mesh = bpy.data.meshes.new("Dreieck_Mesh")
mesh.from_pydata(
    [(0.0, 0.0, 0.0), (2.0, 0.0, 0.0), (0.0, 2.0, 0.0)],
    [],
    [(0, 1, 2)],
)
mesh.update()
obj = bpy.data.objects.new("Dreieck", mesh)
bpy.context.scene.collection.objects.link(obj)
obj["quelle"] = "minimal_bpy_colab.ipynb"

material = bpy.data.materials.new("Blau")
material.diffuse_color = (0.15, 0.35, 0.9, 1.0)
material.use_nodes = True
material.node_tree.nodes["Principled BSDF"].inputs["Base Color"].default_value = material.diffuse_color
mesh.materials.append(material)

output = Path.cwd() / "minimal_bpy_test.blend"
result = bpy.ops.wm.save_as_mainfile(filepath=str(output), compress=True, check_existing=False)
assert result == {"FINISHED"}
print(f"Erstellt: {output} ({output.stat().st_size:,} Byte)")

In [ ]:
# Optional in Google Colab: Datei auf den eigenen Rechner laden.
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    print(output)